# cAPTure: fold-local model preprocessing audit

This CPU-only notebook fits and audits the reviewed 103-column primary packet representation for both development folds. Numeric parameters and variance masks are fitted only on each fold's training scenarios. Validation packets are transformed without refitting. The notebook saves compact JSON preprocessors and reports to Drive; it does not materialize transformed packet tables, build windows, or train a model.


## 1. Mount Drive and load the project


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")

if not PROJECT_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )

required_files = [
    PROJECT_ROOT / "code/python/utils/capture_preprocess.py",
    PROJECT_ROOT / "code/python/tests/test_capture_preprocess.py",
    PROJECT_ROOT / "configs/capture_experiment_v1.yaml",
    PROJECT_ROOT / "configs/capture_packet_schema_v1.yaml",
    PROJECT_ROOT / "configs/capture_preprocessing_v1.yaml",
    PROJECT_ROOT / "code/python/requirements-capture.txt",
]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Update the repository copy first: {missing_files}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))
print("CPU preprocessing-audit environment is ready.")


## 2. Run synthetic contract checks


In [ ]:
test_environment = dict(os.environ)
test_environment["PYTHONPATH"] = str(PROJECT_ROOT / "code/python")
for test_pattern in ("test_capture_feature_profile.py", "test_capture_preprocess.py"):
    subprocess.run(
        [sys.executable, "-m", "unittest", "discover",
         "-s", str(PROJECT_ROOT / "code/python/tests"),
         "-p", test_pattern, "-v"],
        env=test_environment,
        cwd=PROJECT_ROOT,
        check=True,
    )
print("Synthetic preprocessing checks passed.")


## 3. Configure the FULL_DEV audit

The completed canonical preparation run is immutable input. A batch size of 100,000 keeps peak CPU memory bounded while the two fold-specific preprocessors are fitted and verified.


In [ ]:
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display
from utils.capture_preprocess import run_capture_preprocessing_audit

MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
PACKET_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_packet_schema_v1.yaml"
PREPROCESSING_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_preprocessing_v1.yaml"
PREPARED_RUN_DIR = (
    DRIVE_ROOT / "prepared_runs" / "20260917T235058_827743Z_prepare_full_dev"
)
BATCH_SIZE = 100_000

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_preprocessing"
DRIVE_RUN_DIR = DRIVE_ROOT / "preprocessing_runs" / RUN_ID

print(f"Prepared input: {PREPARED_RUN_DIR}")
print(f"Audit output: {DRIVE_RUN_DIR}")


## 4. Fit, transform, and persist the audit

This is the long-running section. Each fold first scans only its training scenarios to fit numeric statistics and the variance mask. It then transforms every development scenario by batches to verify row conservation, fixed width, and finite values. No transformed packet batch is retained.


In [ ]:
AUDIT = run_capture_preprocessing_audit(
    manifest_path=MANIFEST_PATH,
    packet_schema_path=PACKET_SCHEMA_PATH,
    preprocessing_schema_path=PREPROCESSING_SCHEMA_PATH,
    prepared_run_dir=PREPARED_RUN_DIR,
    output_dir=DRIVE_RUN_DIR,
    batch_size=BATCH_SIZE,
)
print(f"Saved preprocessing audit: {DRIVE_RUN_DIR}")


## 5. Review the fixed representation and fold masks


In [ ]:
display(pd.DataFrame({
    "position": range(AUDIT["model_feature_count"]),
    "feature": AUDIT["model_feature_names"],
}))

fold_rows = []
masked_rows = []
numeric_rows = []
for fold, fold_report in AUDIT["folds"].items():
    fold_rows.append({
        "fold": fold,
        "training_scenarios": fold_report["train_scenarios"],
        "validation_scenarios": fold_report["validation_scenarios"],
        "feature_count": fold_report["feature_count"],
        "active_feature_count": fold_report["active_feature_count"],
        "masked_feature_count": fold_report["masked_feature_count"],
    })
    for feature in fold_report["masked_features"]:
        masked_rows.append({"fold": fold, "masked_feature": feature})
    artifact = json.loads(
        (DRIVE_RUN_DIR / fold_report["preprocessor_artifact"]).read_text(encoding="utf-8")
    )
    for feature, parameters in artifact["numeric_parameters"].items():
        numeric_rows.append({"fold": fold, "feature": feature, **parameters})

print("Fold summary")
display(pd.DataFrame(fold_rows))
print("Fold-training masked features")
display(pd.DataFrame(masked_rows))
print("Fold-training numeric parameters")
display(pd.DataFrame(numeric_rows))


## 6. Review scenario-level transformation checks


In [ ]:
scenario_rows = []
for fold, fold_report in AUDIT["folds"].items():
    for scenario, scenario_audit in fold_report["scenario_audits"].items():
        scenario_rows.append({"fold": fold, "scenario": scenario, **scenario_audit})
scenario_table = pd.DataFrame(scenario_rows)
display(scenario_table)

assert AUDIT["model_feature_count"] == 103
assert AUDIT["transformed_packet_artifacts_written"] is False
assert scenario_table["feature_count"].eq(103).all()
assert scenario_table["finite_values"].all()
assert all(
    len(report["train_scenarios"]) + len(report["validation_scenarios"]) == 5
    for report in AUDIT["folds"].values()
)
print("The fold-local preprocessing audit completed without structural errors.")
print("Review the masks and numeric parameters before freezing the contract.")
